# S08-demo-02 — Минимальный тюнинг только на train

Этот ноутбук показывает, как делать **небольшой и честный тюнинг гиперпараметров** без превращения эксперимента в хаотичный перебор.

Логика практикума:

- фиксируем один `train/test split`;
- выбираем **небольшие и осмысленные** сетки гиперпараметров;
- делаем `GridSearchCV` **только на train**;
- test используем один раз — для финальной оценки уже выбранных конфигураций;
- сохраняем артефакты так, чтобы результат можно было воспроизвести.

## Что важно понять

Этот ноутбук не про “выжать максимум любой ценой”.  
Он про инженерную дисциплину:

1. **тюнинг должен быть ограниченным и осмысленным**;
2. **test нельзя использовать как часть подбора**;
3. **улучшение модели нужно фиксировать цифрами и артефактами**.

## 0. Настройки, импорты, пути для артефактов

Сразу фиксируем `RANDOM_STATE`, создаём папку `artifacts/` и подключаем библиотеки.

Мы будем сохранять:

- итоговые метрики на test;
- результаты CV-подбора;
- лучшие параметры;
- лучшую модель;
- JSON-файл с метаданными эксперимента.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET_METRIC = "roc_auc"

NOTEBOOK_DIR = Path.cwd()
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("Рабочая директория:", NOTEBOOK_DIR.resolve())
print("Папка для артефактов:", ARTIFACTS_DIR.resolve())
print("TARGET_METRIC =", TARGET_METRIC)

In [ ]:
def compute_metrics(y_true, y_pred, y_score=None):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    if y_score is not None:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_score))
    else:
        metrics["roc_auc"] = np.nan
    return metrics


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    return compute_metrics(y_test, y_pred, y_score), y_pred, y_score


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def plot_confusion(y_true, y_pred, title):
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
    plt.show()


def summarize_grid_results(grid, model_name, top_n=10):
    cols = [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_fit_time",
        "params",
    ]
    df = pd.DataFrame(grid.cv_results_)[cols].copy()
    df["model"] = model_name
    return df.sort_values(["rank_test_score", "mean_test_score"], ascending=[True, False]).head(top_n)


def make_numeric_preprocessor():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

## 1. Данные и фиксированное разбиение train/test

Возьмём `breast_cancer` из `sklearn`.  
Это удобная бинарная табличная задача, на которой хорошо видно, как аккуратный тюнинг меняет качество, но без тяжёлых вычислений.

Ключевой принцип:

- **split фиксируется один раз**;
- дальше все решения принимаются **только по train**;
- test остаётся нетронутым до финала.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

numeric_features = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Полный размер X:", X.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Доли классов (train):")
print(y_train.value_counts(normalize=True).sort_index())

## 2. Базовые модели до тюнинга

Сначала зафиксируем простые отправные точки:

- `DummyClassifier` — совсем примитивный baseline;
- `LogisticRegression` в pipeline с импутацией и масштабированием;
- `RandomForestClassifier` как базовая деревообразная модель.

Это важно: прежде чем тюнить, надо понимать, **что именно улучшаем**.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

baseline_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logreg_default": Pipeline(
        steps=[
            ("prep", make_numeric_preprocessor()),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
    "random_forest_default": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=1,
            )),
        ]
    ),
}

baseline_rows = []
for name, model in baseline_models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=TARGET_METRIC, n_jobs=None)
    model.fit(X_train, y_train)
    test_metrics, _, _ = evaluate_model(model, X_test, y_test)
    baseline_rows.append({
        "model": name,
        "cv_mean_roc_auc": float(np.mean(cv_scores)),
        "cv_std_roc_auc": float(np.std(cv_scores)),
        **test_metrics,
    })

baseline_df = pd.DataFrame(baseline_rows).sort_values("roc_auc", ascending=False)
baseline_df

Уже на этом этапе видно две вещи:

1. baseline нужен не для галочки, а чтобы понимать масштаб выигрыша;
2. даже до тюнинга разумные модели обычно сильно обгоняют `DummyClassifier`.

## 3. Осмысленные и небольшие сетки гиперпараметров

Здесь важен сам принцип.

Мы не делаем гигантский перебор.  
Мы берём **небольшие сетки**, которые:

- реально можно объяснить;
- реально можно просчитать на обычном ноутбуке;
- дают шанс понять влияние параметров на качество.

Для практикума возьмём две модели:

- `LogisticRegression`: меняем `C` и `class_weight`;
- `RandomForestClassifier`: меняем глубину, число деревьев и минимальный размер листа.

In [ ]:
logreg_pipe = Pipeline(
    steps=[
        ("prep", make_numeric_preprocessor()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]
)

rf_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    ]
)

param_grids = {
    "logreg": {
        "model__C": [0.1, 0.3, 1.0, 3.0],
        "model__class_weight": [None, "balanced"],
    },
    "random_forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 6],
        "model__min_samples_leaf": [1, 2],
    },
}

candidate_models = {
    "logreg": logreg_pipe,
    "random_forest": rf_pipe,
}

## 4. GridSearchCV только на train

Это центральный блок ноутбука.

Для каждой модели:

- подбираем параметры через CV на `X_train, y_train`;
- смотрим лучшие параметры и качество по CV;
- **не трогаем test** до самого конца.

Именно так и должен выглядеть минимальный честный тюнинг.

In [ ]:
grid_objects = {}
grid_top_tables = []
best_params_map = {}

for model_name, model in candidate_models.items():
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        scoring=TARGET_METRIC,
        cv=cv,
        refit=True,
        n_jobs=None,
        return_train_score=False,
    )
    grid.fit(X_train, y_train)
    grid_objects[model_name] = grid
    best_params_map[model_name] = grid.best_params_
    grid_top_tables.append(summarize_grid_results(grid, model_name, top_n=8))

    print(f"=== {model_name} ===")
    print("Лучший CV-score:", round(grid.best_score_, 5))
    print("Лучшие параметры:", grid.best_params_)
    print()

grid_summary_df = pd.concat(grid_top_tables, ignore_index=True)
grid_summary_df

### Краткая интерпретация

Смысл этого вывода не в том, чтобы смотреть только на первую строку.

Нужно обращать внимание на:

- насколько велик выигрыш относительно соседних конфигураций;
- насколько стабилен CV-score;
- не получаем ли мы “случайную победу” на очень маленькой разнице.

## 5. Финальная оценка на test: default vs tuned

Теперь, когда подбор завершён, можно один раз оценить модели на test.

Мы сравним:

- версии **до тюнинга**;
- версии **после тюнинга**.

Так видно, дал ли тюнинг реальный выигрыш, а не только красивые числа внутри CV.

In [ ]:
final_rows = []
fitted_models = {}

# default versions
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    metrics, y_pred, y_score = evaluate_model(model, X_test, y_test)
    fitted_models[name] = model
    final_rows.append({
        "variant": "default",
        "model": name,
        **metrics,
    })

# tuned versions
for name, grid in grid_objects.items():
    best_model = grid.best_estimator_
    metrics, y_pred, y_score = evaluate_model(best_model, X_test, y_test)
    fitted_models[f"{name}_tuned"] = best_model
    final_rows.append({
        "variant": "tuned",
        "model": f"{name}_tuned",
        **metrics,
    })

final_results_df = pd.DataFrame(final_rows).sort_values(["roc_auc", "f1"], ascending=False).reset_index(drop=True)
final_results_df

Хороший практический вопрос после этой таблицы:

**“Тюнинг действительно помог, или выигрыш настолько мал, что инженерно его можно считать несущественным?”**

Для семинара это важнее, чем слепо радоваться любому улучшению в третьем знаке после запятой.

## 6. Насколько улучшение заметно относительно default?

Посчитаем разницу между default- и tuned-версией для тех моделей, которые тюнили.

In [ ]:
comparison_rows = []

pairs = [
    ("logreg_default", "logreg_tuned"),
    ("random_forest_default", "random_forest_tuned"),
]

for default_name, tuned_name in pairs:
    row_default = final_results_df.loc[final_results_df["model"] == default_name].iloc[0]
    row_tuned = final_results_df.loc[final_results_df["model"] == tuned_name].iloc[0]
    comparison_rows.append({
        "family": default_name.replace("_default", ""),
        "delta_accuracy": float(row_tuned["accuracy"] - row_default["accuracy"]),
        "delta_precision": float(row_tuned["precision"] - row_default["precision"]),
        "delta_recall": float(row_tuned["recall"] - row_default["recall"]),
        "delta_f1": float(row_tuned["f1"] - row_default["f1"]),
        "delta_roc_auc": float(row_tuned["roc_auc"] - row_default["roc_auc"]),
    })

delta_df = pd.DataFrame(comparison_rows)
delta_df

В учебных экспериментах часто выясняется полезная вещь:

- тюнинг **может** улучшить результат;
- но улучшение нередко оказывается **умеренным**;
- зато аккуратная постановка эксперимента и выбор разумного семейства моделей влияют сильнее.

Это и есть здоровая инженерная оптика.

## 7. Диагностика лучшей tuned-модели

Выберем лучшую tuned-конфигурацию по `roc_auc` на test и посмотрим confusion matrix.

In [ ]:
tuned_only = final_results_df[final_results_df["variant"] == "tuned"].copy()
best_tuned_name = tuned_only.sort_values(["roc_auc", "f1"], ascending=False).iloc[0]["model"]
best_tuned_model = fitted_models[best_tuned_name]

best_metrics, best_y_pred, best_y_score = evaluate_model(best_tuned_model, X_test, y_test)

print("Лучшая tuned-модель:", best_tuned_name)
print(pd.Series(best_metrics))

plot_confusion(y_test, best_y_pred, title=f"Confusion matrix: {best_tuned_name}")

## 8. Небольшой разбор результата GridSearchCV

Посмотрим на лучшие конфигурации отдельно по семействам моделей.

In [ ]:
for model_name in ["logreg", "random_forest"]:
    print(f"\n=== TOP конфигурации: {model_name} ===")
    display(
        grid_summary_df.loc[grid_summary_df["model"] == model_name]
        .sort_values(["rank_test_score", "mean_test_score"], ascending=[True, False])
        .reset_index(drop=True)
    )

По этим таблицам обычно удобно обсуждать:

- насколько “плотны” результаты;
- есть ли явный лидер;
- стоит ли вообще расширять поиск, или практический выигрыш уже близок к насыщению.

## 9. Сохранение артефактов

Сохраним всё, что нужно для воспроизводимости:

- таблицы результатов;
- лучшие параметры;
- лучшую tuned-модель;
- JSON с метаданными эксперимента.

In [ ]:
baseline_path = ARTIFACTS_DIR / "baseline_metrics.csv"
grid_path = ARTIFACTS_DIR / "grid_search_top_configs.csv"
final_path = ARTIFACTS_DIR / "final_test_metrics.csv"
delta_path = ARTIFACTS_DIR / "tuning_delta.csv"
best_params_path = ARTIFACTS_DIR / "best_params.json"
model_path = ARTIFACTS_DIR / "best_tuned_model.joblib"
meta_path = ARTIFACTS_DIR / "best_tuned_model_meta.json"

baseline_df.to_csv(baseline_path, index=False)
grid_summary_df.to_csv(grid_path, index=False)
final_results_df.to_csv(final_path, index=False)
delta_df.to_csv(delta_path, index=False)
save_json(best_params_map, best_params_path)

joblib.dump(best_tuned_model, model_path)

meta = {
    "target_metric": TARGET_METRIC,
    "random_state": RANDOM_STATE,
    "best_tuned_model_name": best_tuned_name,
    "best_tuned_model_metrics_on_test": best_metrics,
    "best_params_for_all_tuned_models": best_params_map,
    "train_shape": list(X_train.shape),
    "test_shape": list(X_test.shape),
}
save_json(meta, meta_path)

print("Сохранено:")
for path in [baseline_path, grid_path, final_path, delta_path, best_params_path, model_path, meta_path]:
    print("-", path.name)

## 10. Итоги

1. **Минимальный тюнинг надо делать только на train.**  
   Test не должен участвовать в выборе параметров.

2. **Небольшие осмысленные сетки полезнее хаотичного перебора.**  
   Они проще интерпретируются и легче воспроизводятся.

3. **Тюнинг не всегда даёт драматический прирост.**  
   Иногда выигрыш умеренный — и это нормально.

4. **Главная цель — не “победить в таблице”, а получить честный и воспроизводимый результат.**

### Возможное продолжение

Следующий шаг семинара — собрать это в полноценный выбор финальной модели с сохранением всех артефактов эксперимента.